# 第7章：Fine-tuning to Follow Instructions

**目标：** 将预训练好的 GPT 模型通过**指令微调**变成能够遵循指令的模型（类似 ChatGPT）

```
指令微调概述 → 准备指令数据集 → 组织训练批次 → 创建 DataLoader → 加载预训练权重 → 微调训练 → 提取响应 → 评估模型
```

**前置回顾（第2-6章）：**
- 第2章：文本 → Token IDs → DataLoader
- 第3章：Causal Multi-Head Attention
- 第4章：完整 GPT 模型架构
- 第5章：预训练 + 加载 GPT-2 权重 + 文本生成
- 第6章：分类微调——给模型加分类头，输出固定类别
- 现在的问题：预训练模型只会「续写」，分类模型只能输出类别标签 → **如何让模型按指令生成自由文本？** → **指令微调！**

---

## 7.1 指令微调简介 ⭐

### 什么是指令微调？

| 对比项 | 预训练 | 分类微调（Ch6） | 指令微调（Ch7） |
|--------|--------|----------------|----------------|
| 训练目标 | 预测下一个 token | 输出固定类别标签 | 按指令生成自由文本 |
| 训练数据 | 海量无标注文本 | (文本, 标签) 对 | (指令, 响应) 对 |
| 输出形式 | 续写文本 | spam / not spam | 任意文本回答 |
| 代表应用 | GPT-2 | 情感分析、垃圾邮件 | ChatGPT 式对话 |

### 核心思路

- 指令微调 = 在「指令-响应」对数据上继续训练 LLM
- 模型结构**不需要修改**（不像分类微调需要换输出头）
- 训练目标仍然是 next-token prediction，只是训练数据变成了精心设计的「指令格式」文本
- 关键在于数据格式化：将 (instruction, input, output) 拼接为特定模板

> 💡 **关键洞察：** 指令微调不改模型架构，只改训练数据——用「指令→回答」格式的数据，教会模型「看到指令就生成回答」的模式。

---

## 7.2 准备指令数据集 ⭐⭐

我们使用作者准备的 **instruction-data.json** 数据集，包含 1,100 条指令-响应对。

每条数据格式：
```json
{
    "instruction": "任务描述",
    "input": "可选的额外输入",
    "output": "期望的模型回答"
}
```

**数据处理流程：**
1. 下载 JSON 格式的指令数据
2. 使用 **Alpaca 风格模板** 将数据格式化为 LLM 输入
3. 划分训练/验证/测试集（85%/5%/10%）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import os
import json
import re
import time
from tqdm import tqdm

# ─── 第4章的所有组件（直接复用）───────────────────────────────────

GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 768, "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.1, "qkv_bias": False,
}

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads = d_out, num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask[:num_tokens, :num_tokens].bool(), -torch.inf)
        attn_weights = self.dropout(F.softmax(attn_scores / self.head_dim**0.5, dim=-1))
        context_vecs = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vecs)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), GELU(), nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(d_in=cfg["emb_dim"], d_out=cfg["emb_dim"], context_length=cfg["context_length"], num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# ─── 工具函数 ─────────────────────────────────────────
def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={'<|endoftext|>'})).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top_logits[:, -1], torch.tensor(float('-inf')).to(logits.device), logits)
        if temperature > 0.0:
            logits = logits / temperature
            idx_next = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and idx_next.item() == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape 不匹配: {left.shape} vs {right.shape}")
    return nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe.weight"])
    for b in range(len(gpt.trf_blocks)):
        q_w, k_w, v_w = np.split(params[f"h.{b}.attn.c_attn.weight"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)
        q_b, k_b, v_b = np.split(params[f"h.{b}.attn.c_attn.bias"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)
        gpt.trf_blocks[b].att.out_proj.weight = assign(gpt.trf_blocks[b].att.out_proj.weight, params[f"h.{b}.attn.c_proj.weight"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(gpt.trf_blocks[b].att.out_proj.bias, params[f"h.{b}.attn.c_proj.bias"])
        gpt.trf_blocks[b].ff.layers[0].weight = assign(gpt.trf_blocks[b].ff.layers[0].weight, params[f"h.{b}.mlp.c_fc.weight"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(gpt.trf_blocks[b].ff.layers[0].bias, params[f"h.{b}.mlp.c_fc.bias"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(gpt.trf_blocks[b].ff.layers[2].weight, params[f"h.{b}.mlp.c_proj.weight"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(gpt.trf_blocks[b].ff.layers[2].bias, params[f"h.{b}.mlp.c_proj.bias"])
        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale, params[f"h.{b}.ln_1.weight"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift, params[f"h.{b}.ln_1.bias"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale, params[f"h.{b}.ln_2.weight"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift, params[f"h.{b}.ln_2.bias"])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte.weight"])

print("第4-5章组件加载完毕 ✓")

### 下载指令数据集

In [ ]:
import requests

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(response.text)
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"

data = download_and_load_file(file_path, url)
print(f"数据条数: {len(data)}")
print(f"\n带 input 的样本: {data[50]}")
print(f"\n不带 input 的样本: {data[999]}")

### Alpaca 风格模板格式化

我们使用 Alpaca 格式将 (instruction, input, output) 格式化为 LLM 的训练输入。

> 💡 模板提供一致的格式信号，让模型学会「看到 `### Response:` 就开始生成回答」。

In [ ]:
def format_input(entry):
    """将数据条目格式化为 Alpaca 风格的指令输入"""
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

# 查看完整训练样本
model_input = format_input(data[50])
print(model_input + f"\n\n### Response:\n{data[50]['output']}")

In [ ]:
# 划分数据集
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"训练集: {len(train_data)} 条 | 验证集: {len(val_data)} 条 | 测试集: {len(test_data)} 条")

---
## 7.3 组织训练批次 ⭐⭐⭐

核心挑战：
1. 每条样本长度不同 → padding 对齐
2. Target = input 右移一位（next-token prediction）
3. Padding 位置不参与 loss → 用 `ignore_index=-100` 掩码

In [ ]:
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    """自定义 collate：padding + targets + ignore_index 掩码"""
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        inputs_lst.append(inputs)
        targets_lst.append(targets)
    return torch.stack(inputs_lst).to(device), torch.stack(targets_lst).to(device)

# 测试
batch = ([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])
inputs, targets = custom_collate_fn(batch)
print("Inputs:", inputs)
print("Targets:", targets)

---
## 7.4 创建 DataLoader ⭐⭐

In [ ]:
from torch.utils.data import Dataset, DataLoader
from functools import partial

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            full_text = format_input(entry) + f"\n\n### Response:\n{entry['output']}"
            self.encoded_texts.append(tokenizer.encode(full_text))
    def __getitem__(self, index): return self.encoded_texts[index]
    def __len__(self): return len(self.data)

# 选择设备
if torch.cuda.is_available(): device = torch.device("cuda")
elif torch.backends.mps.is_available(): device = torch.device("mps")
else: device = torch.device("cpu")
print(f"使用设备: {device}")

tokenizer = tiktoken.get_encoding("gpt2")
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

torch.manual_seed(123)
batch_size = 8

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=customized_collate_fn, shuffle=True, drop_last=True)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=customized_collate_fn, shuffle=False, drop_last=False)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=customized_collate_fn, shuffle=False, drop_last=False)

print(f"训练集 batch 数: {len(train_loader)} | 验证集: {len(val_loader)} | 测试集: {len(test_loader)}")

---
## 7.5 加载预训练 LLM ⭐⭐

我们使用 **GPT-2 Small (124M)** 进行指令微调。训练速度快，适合学习和实验。

| 模型 | 参数量 | emb_dim | n_layers | n_heads |
|------|--------|---------|----------|--------|
| **gpt2-small** | **124M** | **768** | **12** | **12** |
| gpt2-medium | 355M | 1024 | 24 | 16 |
| gpt2-large | 774M | 1280 | 36 | 20 |
| gpt2-xl | 1558M | 1600 | 48 | 25 |

> 💡 如果想获得更好的指令遵循效果，可将 `CHOOSE_MODEL` 改为 `"gpt2-medium (355M)"`。

In [ ]:
from transformers import GPT2Model as HF_GPT2Model

BASE_CONFIG = {"vocab_size": 50257, "context_length": 1024, "drop_rate": 0.0, "qkv_bias": True}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12, "hf_name": "gpt2"},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16, "hf_name": "gpt2-medium"},
}

# ════════════════════════════════════════════════
# 👇 切换模型只需修改这一行
CHOOSE_MODEL = "gpt2-small (124M)"
# CHOOSE_MODEL = "gpt2-medium (355M)"
# ════════════════════════════════════════════════

hf_name = model_configs[CHOOSE_MODEL].pop("hf_name")
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
print(f"选择模型: {CHOOSE_MODEL} (HuggingFace: {hf_name})")

hf_model = HF_GPT2Model.from_pretrained(hf_name, cache_dir=os.path.join("..", "models", "gpt2"))
hf_state = hf_model.state_dict()

params = {}
params["wte.weight"] = hf_state["wte.weight"].numpy()
params["wpe.weight"] = hf_state["wpe.weight"].numpy()
params["ln_f.weight"] = hf_state["ln_f.weight"].numpy()
params["ln_f.bias"] = hf_state["ln_f.bias"].numpy()
for b in range(BASE_CONFIG["n_layers"]):
    params[f"h.{b}.attn.c_attn.weight"] = hf_state[f"h.{b}.attn.c_attn.weight"].numpy()
    params[f"h.{b}.attn.c_attn.bias"] = hf_state[f"h.{b}.attn.c_attn.bias"].numpy()
    params[f"h.{b}.attn.c_proj.weight"] = hf_state[f"h.{b}.attn.c_proj.weight"].numpy()
    params[f"h.{b}.attn.c_proj.bias"] = hf_state[f"h.{b}.attn.c_proj.bias"].numpy()
    params[f"h.{b}.mlp.c_fc.weight"] = hf_state[f"h.{b}.mlp.c_fc.weight"].numpy()
    params[f"h.{b}.mlp.c_fc.bias"] = hf_state[f"h.{b}.mlp.c_fc.bias"].numpy()
    params[f"h.{b}.mlp.c_proj.weight"] = hf_state[f"h.{b}.mlp.c_proj.weight"].numpy()
    params[f"h.{b}.mlp.c_proj.bias"] = hf_state[f"h.{b}.mlp.c_proj.bias"].numpy()
    params[f"h.{b}.ln_1.weight"] = hf_state[f"h.{b}.ln_1.weight"].numpy()
    params[f"h.{b}.ln_1.bias"] = hf_state[f"h.{b}.ln_1.bias"].numpy()
    params[f"h.{b}.ln_2.weight"] = hf_state[f"h.{b}.ln_2.weight"].numpy()
    params[f"h.{b}.ln_2.bias"] = hf_state[f"h.{b}.ln_2.bias"].numpy()
del hf_model, hf_state

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()
print(f"模型加载完毕 ✓ | 参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 微调前测试
torch.manual_seed(123)
input_text = format_input(val_data[0])
token_ids = generate(model=model, idx=text_to_token_ids(input_text, tokenizer), max_new_tokens=35, context_size=BASE_CONFIG["context_length"], eos_id=50256)
response_text = token_ids_to_text(token_ids, tokenizer)[len(input_text):].replace("### Response:", "").strip()

print(f"指令: {val_data[0]['instruction']}")
print(f"期望: {val_data[0]['output']}")
print(f"模型（微调前）: {response_text}")
print("\n⚠️ 微调前模型不能遵循指令，只是在续写文本")

---
## 7.6 微调训练 ⭐⭐⭐

| 设置 | 值 |
|------|----|
| 优化器 | AdamW (lr=5e-5, weight_decay=0.1) |
| Epochs | 2 |
| 模型 | 由上方 `CHOOSE_MODEL` 决定 |
| 预计耗时 (124M) | CPU ~6 分钟 / MPS ~4 分钟 |
| 预计耗时 (355M) | CPU ~20 分钟 / MPS ~10 分钟 |

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0: return float("nan")
    num_batches = min(num_batches or len(data_loader), len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches: break
        total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
    return total_loss / num_batches

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                    val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
                model.train()
        # 每个 epoch 结束后生成样本
        model.eval()
        token_ids = generate(model=model, idx=text_to_token_ids(start_context, tokenizer).to(device), max_new_tokens=256, context_size=BASE_CONFIG["context_length"], eos_id=50256)
        response = token_ids_to_text(token_ids, tokenizer)[len(start_context):].replace("### Response:", "").strip()
        print(f"  Epoch {epoch+1} 样本: {response[:120]}...")
        model.train()
    return train_losses, val_losses, track_tokens_seen

In [ ]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print(f"训练前 - Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

In [ ]:
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=2, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

print(f"\n训练完成！耗时: {(time.time() - start_time) / 60:.2f} 分钟")

In [ ]:
# 绘制损失曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochs_tensor = torch.linspace(0, 2, len(train_losses))
ax1.plot(epochs_tensor, train_losses, label="训练损失")
ax1.plot(epochs_tensor, val_losses, label="验证损失")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("损失曲线")
ax2.plot(tokens_seen, train_losses, label="训练损失")
ax2.plot(tokens_seen, val_losses, label="验证损失")
ax2.set_xlabel("Tokens seen"); ax2.set_ylabel("Loss"); ax2.legend(); ax2.set_title("损失 vs Token 数")
plt.tight_layout(); plt.show()

---
## 7.7 提取和保存响应 ⭐⭐

In [ ]:
# 查看测试集效果
torch.manual_seed(123)
for entry in test_data[:3]:
    input_text = format_input(entry)
    token_ids = generate(model=model, idx=text_to_token_ids(input_text, tokenizer).to(device), max_new_tokens=256, context_size=BASE_CONFIG["context_length"], eos_id=50256)
    response_text = token_ids_to_text(token_ids, tokenizer)[len(input_text):].replace("### Response:", "").strip()
    print(f"指令: {entry['instruction']}")
    print(f"✅ 期望: {entry['output']}")
    print(f"🤖 模型: {response_text}")
    print("─" * 50)

In [ ]:
# 全测试集生成并保存
for i, entry in tqdm(enumerate(test_data), total=len(test_data), desc="生成测试集回答"):
    input_text = format_input(entry)
    token_ids = generate(model=model, idx=text_to_token_ids(input_text, tokenizer).to(device), max_new_tokens=256, context_size=BASE_CONFIG["context_length"], eos_id=50256)
    test_data[i]["model_response"] = token_ids_to_text(token_ids, tokenizer)[len(input_text):].replace("### Response:", "").strip()

with open("instruction-data-with-response.json", "w") as f:
    json.dump(test_data, f, indent=4)

# 保存模型
file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL)}-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"\n模型已保存: {file_name}")

---
## 7.8 评估微调模型（可选）⭐⭐

使用智谱 AI 的 **GLM-4-Flash** 作为 LLM-as-Judge 来评分。该模型完全免费，速度快（~72 token/s），支持 128K 上下文。

**前置要求：**
安装 SDK：`pip install zhipuai`

In [ ]:
from zhipuai import ZhipuAI

ZHIPU_API_KEY = "d63eddac939145a1a2564bfb51a58ac9.OM0UPDTi3lBFNMcu"  # 可以直接使用
EVAL_MODEL = "glm-4-flash"

client = ZhipuAI(api_key=ZHIPU_API_KEY)

test_resp = client.chat.completions.create(
    model=EVAL_MODEL,
    messages=[{"role": "user", "content": "回复数字 42"}],
    temperature=0.0,
    max_tokens=10,
)
print(f"GLM-4-Flash 连接测试: {test_resp.choices[0].message.content.strip()} ✓")

In [ ]:
def query_model(prompt):
    """调用 GLM-4-Flash API 进行评分"""
    response = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

scores = []
failed = 0
for i, entry in tqdm(enumerate(test_data), total=len(test_data), desc="GLM-4-Flash 评估中"):
    prompt = (f"Given the input `{format_input(entry)}` and correct output `{entry['output']}`, "
              f"score the model response `{entry['model_response']}` on a scale from 0 to 100, "
              f"where 100 is the best score. Respond with the integer number only.")
    raw_response = query_model(prompt)
    try:
        score = int(re.search(r'\d+', raw_response).group())
        scores.append(score)
    except (ValueError, AttributeError):
        failed += 1

print(f"\n平均分: {sum(scores)/len(scores):.2f} / 100")
print(f"有效评分: {len(scores)}/{len(test_data)} | 解析失败: {failed}")

---
## 7.9 总结 ⭐

### 全书学习路线（已完成！）

```
Ch2-4: 实现 GPT 模型架构
Ch5:   预训练（无标注数据，学语言规律）
Ch6:   分类微调（有标签，输出类别）
Ch7:   指令微调（指令-响应对，输出自由文本） ← 你在这里!
```

### 本章关键要点
- 指令微调**不改模型架构**，只改训练数据格式
- 使用 Alpaca 模板格式化 (instruction, input, output)
- Collate 函数实现动态 padding + ignore_index 掩码
- 训练仍是 next-token prediction
- 评估可用 LLM-as-Judge（如 Llama 3）

### ✏️ 练习
1. 修改为 Phi-3 风格模板，比较效果
2. 掩码指令部分（只在响应部分计算 loss）
3. 用 gpt2-medium (355M) 重新训练，对比效果
4. 尝试不同学习率 (1e-5, 5e-5, 1e-4)

**恭喜完成全书学习！** 🎉